# 24. Reviewer closers: two missing baselines, intervals on every headline number, and the judge validation sheet

Four things a referee asked for, none of them a new experiment.

1. **PIGuard** [Li et al., ACL 2025], the guard aimed at over-defense, was skipped earlier because its repository ships custom model code. Stage B downloads that file, prints it in full, and scores the model only after `ALLOW_REMOTE_CODE` is set to True by hand, so the decision to run it is a read decision, not a prompt.
2. **A conventional fine-tune** of the same backbone (DeBERTa-v3-base) on public prompt-injection data, with no matched pairs and no document hosts, so the question "is this just fine-tuning DeBERTa?" has a direct answer. Trained on a public injection corpus that is not one of the evaluation sets; if that corpus cannot be loaded, it falls back to the deepset and jailbreak training splits and marks the two contaminated columns.
3. **Intervals.** Item-bootstrap intervals for every AUROC in the paper, Wilson intervals for every rate over a small count (manufactured misses, the transfer matrix, the category table), and, for the interval report, the sampling interval at each endpoint next to the identification gap so the two quantities are visibly different.
4. **The judge validation sheet.** A stratified sample of 100 judged items from the Phase 5 rerun, written as a CSV with the instruction and the target's output and an empty `human_label` column; the judge's verdict is kept in a separate file so labeling is blind. Stage F reads the filled sheet and reports agreement and Cohen's kappa.

GPU for stages B and C (scoring, one short fine-tune); the rest runs from cached files.

**Outputs.** `reports/guard_panel_extended_ci.csv`, `reports/panel_interval_ci.csv`, `reports/structural_detector_eval_ci.csv`, `reports/phase5v2_manufactured_ci.csv`, `reports/phase5v2_transfer_ci.csv`, `reports/phase5v2_by_category_ci.csv`, `reports/judge_validation_sheet.csv`, `reports/judge_validation.csv`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
!pip install -q transformers datasets accelerate sentencepiece
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull

os.makedirs('reports', exist_ok=True); os.makedirs('data/closers', exist_ok=True); os.makedirs('data/structural/scores', exist_ok=True)
import torch, gc; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Stage A. Module and frames

In [ ]:
%%writefile src/ci_tools.py
"""Uncertainty for the paper's headline numbers: item bootstrap for AUROC, Wilson intervals for rates, and the
identification gap side by side with the sampling interval at each endpoint."""
from __future__ import annotations
import numpy as np


def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (max(0.0, (c - h) / d), min(1.0, (c + h) / d))


def boot_auroc(y, p, B=2000, seed=0):
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]; v = []
    for _ in range(B):
        a = rng.choice(pos, len(pos), replace=True); b = rng.choice(neg, len(neg), replace=True)
        v.append(roc_auc_score(np.r_[np.ones(len(a)), np.zeros(len(b))], np.r_[p[a], p[b]]))
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


def boot_rate(x, B=2000, seed=0):
    rng = np.random.default_rng(seed); x = np.asarray(x, float); v = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


def fmt(v, lo, hi, d=3):
    return f"{v:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"


In [ ]:
import importlib, json, numpy as np, pandas as pd
import ci_tools, data_loaders, interval_panel, structural_train; [importlib.reload(m) for m in (ci_tools, data_loaders, interval_panel, structural_train)]
from ci_tools import wilson, boot_auroc, boot_rate, fmt
from data_loaders import load_deepset, load_bipia_categorized, load_notinject, load_jailbreak, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from interval_panel import three_thresholds, fnr, fpr, KEYS
from structural_train import score_texts, load_jsonl, train_model
from detectors import score_released
from sklearn.metrics import roc_auc_score
deepset = load_deepset(); bip = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS); noti = load_notinject(); jb = load_jailbreak()
OPI = json.load(open('data/opi/opi_eval.json')); OTXT = [r['text'] for r in OPI['records']]; OY = np.array([r['label'] for r in OPI['records']])
yd = deepset.label.values; HOST = bip.label.values == 0; HIJ = bip.meta.isin(BIPIA_HIJACK).values; HARM = bip.meta.isin(BIPIA_HARMFUL).values
SETS = {'bipia': bip.text.tolist(), 'notinject': noti.text.tolist(), 'opi': OTXT, 'deepset': deepset.text.tolist(), 'jailbreak': jb.text.tolist()}
HOW = json.load(open('data/closers/howto.json')); SETS['howto'] = HOW['texts']
AD = json.load(open('data/agentdojo/set.json')) if os.path.exists('data/agentdojo/set.json') else None
if AD: SETS['agentdojo'] = [r['text'] for r in AD]
print('sets:', {k: len(v) for k, v in SETS.items()})

## Stage B. PIGuard, after reading its code

Run once with `ALLOW_REMOTE_CODE = False`: it downloads and prints `modeling_piguard.py`. Read it. If it is what a model class should be (a classification head on a DeBERTa encoder and nothing else, no network calls, no file writes, no subprocess), set the flag to True and run again.

In [ ]:
ALLOW_REMOTE_CODE = True
PIG_ID = 'leolee99/PIGuard'
from huggingface_hub import hf_hub_download
try:
    fpth = hf_hub_download(PIG_ID, 'modeling_piguard.py'); code = open(fpth).read()
    print(f'--- {PIG_ID}/modeling_piguard.py ({len(code)} chars) ---'); print(code); print('--- end ---')
    suspicious = [w for w in ('import os', 'subprocess', 'requests', 'urllib', 'socket', 'open(', 'exec(', 'eval(', 'pickle', 'base64') if w in code]
    print('patterns worth a second look:', suspicious or 'none')
except Exception as e: print('could not fetch the file:', type(e).__name__, str(e)[:160])
if ALLOW_REMOTE_CODE:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    try: tok = AutoTokenizer.from_pretrained(PIG_ID, trust_remote_code=True)
    except Exception as e: print('PIGuard tokenizer failed (', type(e).__name__, '); using the base DeBERTa-v3 tokenizer it was fine-tuned from'); tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
    try: mdl = AutoModelForSequenceClassification.from_pretrained(PIG_ID, trust_remote_code=True).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
    except Exception as e: raise SystemExit(f'PIGuard model failed to load: {type(e).__name__}: {str(e)[:200]}')
    print('id2label:', mdl.config.id2label)
    idx = next((int(i) for i, l in mdl.config.id2label.items() if any(h in str(l).lower() for h in ('inject', 'malicious', 'label_1'))), mdl.config.num_labels - 1); print('attack class index:', idx)
    def score_pig(texts, bs=16):
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), bs):
                b = tok(texts[i:i + bs], truncation=True, max_length=512, padding=True, return_tensors='pt').to(mdl.device); out.append(torch.softmax(mdl(**b).logits.float(), -1)[:, idx].cpu().numpy())
        return np.concatenate(out)
    for s, texts in SETS.items():
        fp = f'data/closers/score_piguard_{s}.npy' if s != 'agentdojo' else 'data/agentdojo/score_piguard.npy'
        if not os.path.exists(fp): print('scoring PIGuard on', s); np.save(fp, score_pig(texts))
    json.dump(PIG_ID, open('data/closers/id_piguard.json', 'w')); del mdl; gc.collect(); torch.cuda.empty_cache(); print('PIGuard scored and cached')
else:
    print('ALLOW_REMOTE_CODE is False: PIGuard not run. Read the code above, then set it to True and re-run this cell.')

## Stage C. Conventional fine-tune baseline

DeBERTa-v3-base fine-tuned on a public prompt-injection corpus with the same recipe as the structural detector (two epochs, class-weighted loss), but with ordinary injection prompts as positives and ordinary prompts as negatives: no document hosts, no matched pairs. Trained on a public corpus that is not one of the evaluation sets when one loads; otherwise on the deepset and jailbreak training splits, in which case the direct and jailbreak columns are contaminated and are marked.

In [ ]:
best = 'data/structural/model_conventional_ft/best'
CONV_SOURCE = json.load(open('data/structural/conventional_source.json')) if os.path.exists('data/structural/conventional_source.json') else None
if os.path.exists(os.path.join(best, 'config.json')): print('conventional baseline exists:', CONV_SOURCE)
else:
    from datasets import load_dataset
    recs, src = [], None
    for cid, tf, lf, pos_vals in [('xTRam1/safe-guard-prompt-injection', 'text', 'label', {1, '1', 'injection', True}), ('Lakera/gandalf_ignore_instructions', 'text', None, None)]:
        try:
            ds = load_dataset(cid, split='train')
            if lf is None: continue
            for r in ds:
                recs.append({'text': str(r[tf]), 'label': 1 if r[lf] in pos_vals else 0})
            src = cid; break
        except Exception as e: print(cid, 'not loaded:', type(e).__name__)
    if not recs:
        d1 = load_dataset('deepset/prompt-injections', split='train'); d2 = load_dataset('jackhhao/jailbreak-classification', split='train')
        recs = [{'text': str(r['text']), 'label': int(r['label'])} for r in d1] + [{'text': str(r['prompt']), 'label': 1 if str(r['type']).lower() == 'jailbreak' else 0} for r in d2]
        src = 'deepset train + jailbreak train (CONTAMINATES direct and jailbreak columns)'
    import random; random.Random(0).shuffle(recs); n = len(recs); val = recs[: max(100, n // 10)]; tr = recs[len(val):]
    print(f'training conventional baseline on {src}: {len(tr)} train ({sum(r["label"] for r in tr)} positive), {len(val)} val')
    train_model(tr, val, 'data/structural/model_conventional_ft', seed=0, epochs=2)
    json.dump(src, open('data/structural/conventional_source.json', 'w')); CONV_SOURCE = src
for s, texts in SETS.items():
    fp = f'data/structural/scores/conventional_ft_{s}.npy' if s not in ('howto', 'agentdojo') else (f'data/closers/score_conventional_ft_howto.npy' if s == 'howto' else 'data/agentdojo/score_conventional_ft.npy')
    if not os.path.exists(fp): print('scoring conventional baseline on', s); np.save(fp, score_texts(best, texts))
print('conventional baseline scored; source:', CONV_SOURCE)

## Stage D. Intervals for every headline table (cached scores only)

In [ ]:
KEY = {'bipia': 'bipia_cat', 'notinject': 'notinject', 'deepset': 'deepset', 'jailbreak': 'jailbreak'}
REL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
EXTRA = {'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert'}
S = {}
for t, nm in REL.items(): S[nm] = {s: np.load(f'data/score_{t}_{KEY[s]}.npy') for s in KEY}; S[nm]['opi'] = np.load(f'data/opi/score_{t}.npy')
for t, nm in EXTRA.items():
    if all(os.path.exists(f'data/closers/score_{t}_{s}.npy') for s in ('bipia', 'notinject', 'opi', 'deepset', 'jailbreak')): S[nm] = {s: np.load(f'data/closers/score_{t}_{s}.npy') for s in ('bipia', 'notinject', 'opi', 'deepset', 'jailbreak')}
if os.path.exists('data/closers/score_piguard_bipia.npy'): S['PIGuard'] = {s: np.load(f'data/closers/score_piguard_{s}.npy') for s in ('bipia', 'notinject', 'opi', 'deepset', 'jailbreak')}
for cfg, nm in (('full', 'structural (full)'), ('full_aug', 'structural (full_aug)'), ('ablation_no_twins', 'structural (ablation_no_twins)'), ('full_sizematched', 'structural (full_sizematched)'), ('conventional_ft', 'conventional fine-tune')):
    fps = {s: f'data/structural/scores/{cfg}_{s}.npy' for s in ('bipia', 'notinject', 'deepset', 'jailbreak')}; fps['opi'] = f'data/opi/score_structural_{cfg}.npy' if cfg != 'conventional_ft' else 'data/structural/scores/conventional_ft_opi.npy'
    if all(os.path.exists(f) for f in fps.values()): S[nm] = {s: np.load(f) for s, f in fps.items()}
# polarity check for guards whose label order the loader may have guessed
for nm in list(S):
    if nm in ('deepset injection', 'fmops distilbert', 'ProtectAI-v1', 'PIGuard') and roc_auc_score(yd, S[nm]['deepset']) < 0.5: S[nm] = {k: 1 - v for k, v in S[nm].items()}; print(nm, 'inverted')
yh = np.r_[np.ones(int(HIJ.sum())), np.zeros(int(HOST.sum()))]; yr = np.r_[np.ones(int(HARM.sum())), np.zeros(int(HOST.sum()))]
rows = []
for nm, sc in S.items():
    pb = sc['bipia']; hosts = pb[HOST]; t = three_thresholds(hosts)['guar']; po = sc['opi']; to = three_thresholds(po[OY == 0])['guar']
    au = roc_auc_score(yh, np.r_[pb[HIJ], hosts]); alo, ahi = boot_auroc(yh, np.r_[pb[HIJ], hosts])
    ah = roc_auc_score(yr, np.r_[pb[HARM], hosts]); hlo, hhi = boot_auroc(yr, np.r_[pb[HARM], hosts])
    ao = roc_auc_score(OY, po); olo, ohi = boot_auroc(OY, po)
    m = pb[HIJ] < t; wlo, whi = wilson(int(m.sum()), len(m)); mo = po[OY == 1] < to; olo2, ohi2 = wilson(int(mo.sum()), len(mo))
    rows.append(dict(model=nm, AUROC_hijack=fmt(au, alo, ahi), AUROC_harmful=fmt(ah, hlo, hhi), FNR_hijack_doc=fmt(m.mean(), wlo, whi), notinject_FPR_doc=fmt(fpr(sc['notinject'], t), *wilson(int((sc['notinject'] >= t).sum()), len(sc['notinject']))),
                     AUROC_second_source=fmt(ao, olo, ohi), FNR_second_source=fmt(mo.mean(), olo2, ohi2), AUROC_direct=f'{roc_auc_score(yd, sc["deepset"]):.3f}', AUROC_jailbreak=f'{roc_auc_score(jb.label.values, sc["jailbreak"]):.3f}'))
PANEL = pd.DataFrame(rows); PANEL.to_csv('reports/guard_panel_extended_ci.csv', index=False)
pd.set_option('display.width', 300); print('=== extended panel with 95 percent intervals ==='); print(PANEL.to_string(index=False))
# identification gap next to sampling intervals (from the notebook-15 report, which stored bootstrap CIs at each endpoint)
PI = pd.read_csv('reports/panel_interval.csv'); PI = PI[PI['shift'] != 'over_defense']
ID = pd.DataFrame(dict(detector=PI.detector, shift=PI['shift'], identification_gap=(PI.FNR_hi - PI.FNR_lo).round(3), FNR_low_end=[fmt(a, b, c) for a, b, c in zip(PI.FNR_lo, PI.FNR_lo_lo95, PI.FNR_lo_hi95)], FNR_high_end=[fmt(a, b, c) for a, b, c in zip(PI.FNR_hi, PI.FNR_hi_lo95, PI.FNR_hi_hi95)], sampling_half_width_low=((PI.FNR_lo_hi95 - PI.FNR_lo_lo95) / 2).round(3)))
ID.to_csv('reports/panel_interval_ci.csv', index=False); print('\n=== identification gap versus sampling interval at each endpoint ==='); print(ID.to_string(index=False))
# Phase 5 rates with Wilson intervals
M = pd.read_csv('reports/phase5v2_manufactured.csv'); M['manufactured_ci'] = [fmt(r.manufactured_rate, *wilson(int(r.n_manufactured), int(r.n))) for _, r in M.iterrows()]; M['still_works_ci'] = [fmt(r.still_works, *wilson(int(round(r.still_works * r.n)), int(r.n))) for _, r in M.iterrows()]
M.to_csv('reports/phase5v2_manufactured_ci.csv', index=False); print('\n=== manufactured misses with Wilson intervals ==='); print(M[['detector', 'threshold', 'still_works_ci', 'manufactured_ci']].to_string(index=False))
T = pd.read_csv('reports/phase5v2_transfer.csv'); cols = [c for c in T.columns if c.startswith('caught_by_')]
for c in cols: T[c + '_ci'] = [fmt(v, *wilson(int(round(v * n)), int(n))) if not np.isnan(v) else 'n/a' for v, n in zip(T[c], T.n_manufactured)]
T.to_csv('reports/phase5v2_transfer_ci.csv', index=False); print('\n=== transfer with Wilson intervals ==='); print(T[['source_detector', 'n_manufactured'] + [c + '_ci' for c in cols]].to_string(index=False))
BC = pd.read_csv('reports/phase5v2_by_category.csv'); mc = [c for c in BC.columns if c.startswith('manufactured_')]
for c in mc: BC[c + '_ci'] = [fmt(v, *wilson(int(round(v * n)), int(n))) for v, n in zip(BC[c], BC.n)]
BC.to_csv('reports/phase5v2_by_category_ci.csv', index=False); print('\n=== by category with Wilson intervals (n = 13 per cell) ==='); print(BC[['category'] + [c + '_ci' for c in mc]].to_string(index=False))

## Stage E. Judge validation sheet

Writes 100 judged items (stratified by source and by verdict) with the instruction, the target's output and an empty `human_label` column. Label each row 1 if the output follows the injected instruction, 0 if it does not, then run Stage F. The judge's verdicts are kept in a separate file so labeling is blind.

In [ ]:
sheet = 'reports/judge_validation_sheet.csv'
if os.path.exists(sheet): print('sheet exists; not overwriting. Fill human_label, then run Stage F.')
else:
    SUB = pd.DataFrame(json.load(open('data/phase5v2/subset.json'))); OUTS = json.load(open('data/phase5v2/exec_out.json')); JV = json.load(open('data/phase5v2/judge_7b.json'))
    items = [(k, bool(v)) for k, v in JV.items()]
    rng = np.random.default_rng(0); pos = [k for k, v in items if v]; neg = [k for k, v in items if not v]
    pick = list(rng.choice(pos, min(50, len(pos)), replace=False)) + list(rng.choice(neg, min(100 - min(50, len(pos)), len(neg)), replace=False)); rng.shuffle(pick)
    rows = []
    for k in pick:
        src, i = k.split('|'); i = int(i)
        rows.append(dict(item_id=k, source=src, category=SUB.meta.iloc[i], instruction=SUB.instruction.iloc[i], target_output=OUTS[src][i], human_label=''))
    pd.DataFrame(rows).to_csv(sheet, index=False)
    pd.DataFrame([dict(item_id=k, judge_label=int(JV[k])) for k in pick]).to_csv('data/phase5v2/judge_validation_key.csv', index=False)
    print('sheet written:', sheet, '| 100 items; judge verdicts kept in data/phase5v2/judge_validation_key.csv')

## Stage F. Agreement between the judge and the human labels (run after filling the sheet)

In [ ]:
H = pd.read_csv('reports/judge_validation_sheet.csv'); K = pd.read_csv('data/phase5v2/judge_validation_key.csv')
H = H.merge(K, on='item_id'); done = H[H.human_label.notna() & (H.human_label.astype(str).str.strip() != '')]
if len(done) < 20: print(f'only {len(done)} rows labeled; fill the sheet and re-run')
else:
    h = done.human_label.astype(int).values; j = done.judge_label.astype(int).values
    agree = float((h == j).mean()); po = agree; pe = float(((h == 1).mean() * (j == 1).mean()) + ((h == 0).mean() * (j == 0).mean())); kappa = (po - pe) / (1 - pe) if pe < 1 else np.nan
    tp = int(((h == 1) & (j == 1)).sum()); fpn = int(((h == 0) & (j == 1)).sum()); fnn = int(((h == 1) & (j == 0)).sum()); tn = int(((h == 0) & (j == 0)).sum())
    JVAL = pd.DataFrame([dict(n=len(done), agreement=agree, agreement_lo95=wilson(int((h == j).sum()), len(done))[0], agreement_hi95=wilson(int((h == j).sum()), len(done))[1], cohen_kappa=kappa, judge_precision=tp / max(1, tp + fpn), judge_recall=tp / max(1, tp + fnn), tp=tp, fp=fpn, fn=fnn, tn=tn)])
    JVAL.to_csv('reports/judge_validation.csv', index=False); print(JVAL.round(3).to_string(index=False))

## Stage G. Log and commit

In [ ]:
from reslog import log_result
summary = 'Extended panel with CIs:\n' + PANEL.to_string(index=False) + '\n\nIdentification gap vs sampling:\n' + ID.to_string(index=False)
if os.path.exists('reports/judge_validation.csv'): summary += '\n\nJudge validation:\n' + pd.read_csv('reports/judge_validation.csv').round(3).to_string(index=False)
log_result('nb24: PIGuard, conventional fine-tune baseline, intervals on headline numbers, judge validation sheet', summary, csv_df=PANEL, csv_name='guard_panel_extended_ci.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb24: PIGuard (code read first), conventional fine-tune baseline, bootstrap and Wilson intervals for every headline table, judge validation sheet; add src/ci_tools.py"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)